# ByteCNN — Sequential Year-by-Year Training (Hugging Face APK Bucket)

Trains the **1D-CNN (ByteCNN)** on the [sakhawat2088/android-apks](https://huggingface.co/buckets/sakhawat2088/android-apks) **HF Bucket** (~108 GB, ~13.5k APKs).

> **Important:** This data lives in a **Bucket** (`hf://buckets/...`), **not** a Model/Dataset repo. The notebook uses `list_bucket_tree` + `download_bucket_files` (requires `huggingface_hub>=1.10`).

**Strategy**
- Train **2020 → 2021 → 2022 → 2023** (chronological continual learning)
- After each year: **load previous checkpoint**, fine-tune, save updated weights
- **Stream downloads** in batches (full bucket does not fit on Kaggle disk)
- Output: **year-wise CSV stats**, final `.pth`, embedded `.onnx` for VigiDroid

**Kaggle setup**
1. Settings → Accelerator → **GPU** (recommended)
2. Settings → Internet → **On**
3. **HF_TOKEN not required** for public buckets (optional if access fails)
4. Start with `MAX_PER_CLASS=200`, then set `None` for full training

In [ ]:
!pip install -q "huggingface_hub>=1.10" onnx onnxscript tqdm

In [ ]:
import gc
import os
import random
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import download_bucket_files, list_bucket_tree
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK = Path("/kaggle/working/bytecnn")
CACHE = WORK / "hf_cache"
MODELS = WORK / "models"
STATS = WORK / "stats"
for p in (WORK, CACHE, MODELS, STATS):
    p.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────

# HF *Bucket* id (NOT a model/dataset repo_id)
# https://huggingface.co/buckets/sakhawat2088/android-apks
BUCKET_ID = "sakhawat2088/android-apks"

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

YEAR_ORDER = ["2020", "2021", "2022", "2023"]

BYTE_LENGTH = 1024
FROM_END = True
BATCH_SIZE = 64
EPOCHS_PER_YEAR = 10
LR_FIRST_YEAR = 1e-3
LR_NEXT_YEARS = 3e-4
VAL_FRACTION = 0.1
DOWNLOAD_CHUNK = 400
MAX_PER_CLASS: Optional[int] = 500

CHECKPOINT_NAME = "bytecnn_continual.pth"
ONNX_NAME = "bytecnn_basemodel_2020.onnx"

In [ ]:
class ByteCNN(nn.Module):
    """Same architecture as 1dcnn/src/model/bytecnn.py (VigiDroid-compatible)."""

    def __init__(self, embed_dim: int = 8, num_classes: int = 2) -> None:
        super().__init__()
        self.embed = nn.Embedding(256, embed_dim)
        self.conv1 = nn.Conv1d(embed_dim, 32, kernel_size=5)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 32, kernel_size=5)
        self.bn2 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(kernel_size=5, stride=5)
        self.conv3 = nn.Conv1d(32, 32, kernel_size=5)
        self.bn3 = nn.BatchNorm1d(32)
        self.conv4 = nn.Conv1d(32, 32, kernel_size=5)
        self.bn4 = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(kernel_size=5, stride=5)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embed(x).transpose(1, 2)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = x.mean(dim=2)
        return self.fc(x)


def read_tail_bytes(path: Path, length: int = 1024, from_end: bool = True) -> torch.Tensor:
    with open(path, "rb") as f:
        if from_end:
            try:
                f.seek(-length, 2)
                segment = f.read(length)
            except OSError:
                f.seek(0)
                segment = f.read().rjust(length, b"\0")
        else:
            segment = f.read(length).ljust(length, b"\0")
    return torch.tensor(list(segment), dtype=torch.long)


class APKPathDataset(Dataset):
    def __init__(self, samples: List[Tuple[Path, int]], byte_length=1024, from_end=True):
        self.samples = samples
        self.byte_length = byte_length
        self.from_end = from_end

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        x = read_tail_bytes(path, self.byte_length, self.from_end)
        return x, label

In [ ]:
def _list_prefix(prefix: str) -> List[str]:
    """List .apk paths under a bucket prefix using HF Buckets API."""
    paths = []
    kwargs = {"recursive": True, "prefix": prefix}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN

    for item in list_bucket_tree(BUCKET_ID, **kwargs):
        item_type = getattr(item, "type", None)
        path = getattr(item, "path", None)
        if path is None:
            continue
        if item_type == "directory":
            continue
        if path.endswith(".apk"):
            paths.append(path)

    if not paths:
        raise RuntimeError(
            f"No .apk files under bucket prefix '{prefix}'. "
            f"Bucket={BUCKET_ID}. Check BUCKET_ID and internet access."
        )
    print(f"  listed {len(paths)} apks under {prefix}")
    return paths


def list_year_apk_paths(year: str) -> Tuple[List[str], List[str]]:
    benign = _list_prefix(f"{year}/benign")
    malware = _list_prefix(f"{year}/malware")
    benign.sort()
    malware.sort()
    if MAX_PER_CLASS is not None:
        benign = benign[:MAX_PER_CLASS]
        malware = malware[:MAX_PER_CLASS]
    return benign, malware


def download_apk(remote_path: str, dest_dir: Path) -> Path:
    dest_dir.mkdir(parents=True, exist_ok=True)
    dst = dest_dir / Path(remote_path).name
    kwargs = {}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    download_bucket_files(
        BUCKET_ID,
        files=[(remote_path, str(dst))],
        **kwargs,
    )
    return dst


def download_paths(remote_paths: List[str], dest_dir: Path) -> List[Path]:
    local_paths = []
    for rp in tqdm(remote_paths, desc=f"Download {dest_dir.name}", leave=False):
        try:
            local_paths.append(download_apk(rp, dest_dir))
        except Exception as ex:
            print(f"WARN skip {rp}: {ex}")
    return local_paths


def cleanup_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)
    gc.collect()


# Quick connectivity test (run once before full training)
try:
    _probe = _list_prefix("2020/benign")
    print(f"Bucket OK: {BUCKET_ID} — sample: {_probe[0]}")
except Exception as e:
    print("Bucket probe FAILED:", e)
    print("Ensure huggingface_hub>=1.10 and Internet is ON.")

In [ ]:
def train_epoch(model, loader, optimizer) -> float:
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y).item()
        correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    acc = correct / max(n, 1)
    loss = total_loss / max(len(loader), 1)
    return acc, loss

In [ ]:
@dataclass
class YearStats:
    year: str
    benign_total: int
    malware_total: int
    total_samples: int
    val_samples: int
    train_chunks: int
    epochs: int
    best_val_accuracy: float
    best_val_loss: float
    final_train_loss: float
    final_val_loss: float
    final_val_accuracy: float
    learning_rate: float
    continued_from_checkpoint: bool


def train_one_year(
    model: ByteCNN,
    year: str,
    checkpoint: Path,
    *,
    continued: bool,
) -> YearStats:
    lr = LR_NEXT_YEARS if continued else LR_FIRST_YEAR
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    benign_paths, malware_paths = list_year_apk_paths(year)
    all_remote = [(p, 0) for p in benign_paths] + [(p, 1) for p in malware_paths]
    random.shuffle(all_remote)

    n_val = max(1, int(len(all_remote) * VAL_FRACTION))
    val_remote = all_remote[:n_val]
    train_remote = all_remote[n_val:]

    year_work = WORK / "tmp" / year
    val_dir = year_work / "val"
    cleanup_dir(val_dir)

    print(f"\n{'='*60}\nYear {year}: {len(benign_paths)} benign + {len(malware_paths)} malware")
    print(f"Train pool={len(train_remote)}  Val={len(val_remote)}  LR={lr}")

    # Download validation set once
    val_samples = []
    for rp, label in tqdm(val_remote, desc="Val download"):
        lp = download_apk(rp, val_dir / ("benign" if label == 0 else "malware"))
        val_samples.append((lp, label))
    val_loader = DataLoader(
        APKPathDataset(val_samples, BYTE_LENGTH, FROM_END),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    best_val_acc = -1.0
    best_val_loss = float("inf")
    final_train_loss = 0.0
    final_val_loss = 0.0
    final_val_acc = 0.0
    chunk_count = 0

    for epoch in range(EPOCHS_PER_YEAR):
        random.shuffle(train_remote)
        epoch_train_losses = []

        for start in range(0, len(train_remote), DOWNLOAD_CHUNK):
            chunk = train_remote[start : start + DOWNLOAD_CHUNK]
            chunk_dir = year_work / f"train_chunk_{start}"
            cleanup_dir(chunk_dir)
            chunk_count += 1

            train_samples = []
            for rp, label in chunk:
                lp = download_apk(rp, chunk_dir / ("benign" if label == 0 else "malware"))
                train_samples.append((lp, label))

            if not train_samples:
                continue

            train_loader = DataLoader(
                APKPathDataset(train_samples, BYTE_LENGTH, FROM_END),
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=0,
            )
            loss = train_epoch(model, train_loader, optimizer)
            epoch_train_losses.append(loss)
            cleanup_dir(chunk_dir)

        final_train_loss = float(np.mean(epoch_train_losses)) if epoch_train_losses else 0.0
        val_acc, val_loss = evaluate(model, val_loader)
        final_val_acc, final_val_loss = val_acc, val_loss

        print(
            f"  Epoch {epoch+1}/{EPOCHS_PER_YEAR} | train_loss={final_train_loss:.4f} "
            f"| val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint)
            print(f"    saved checkpoint (val_acc={val_acc:.4f})")

    cleanup_dir(year_work / "val")
    cleanup_dir(year_work)

    if not checkpoint.is_file():
        torch.save(model.state_dict(), checkpoint)

    return YearStats(
        year=year,
        benign_total=len(benign_paths),
        malware_total=len(malware_paths),
        total_samples=len(benign_paths) + len(malware_paths),
        val_samples=len(val_remote),
        train_chunks=chunk_count,
        epochs=EPOCHS_PER_YEAR,
        best_val_accuracy=best_val_acc,
        best_val_loss=best_val_loss,
        final_train_loss=final_train_loss,
        final_val_loss=final_val_loss,
        final_val_accuracy=final_val_acc,
        learning_rate=lr,
        continued_from_checkpoint=continued,
    )

In [ ]:
def export_onnx(checkpoint: Path, out_path: Path) -> None:
    import onnx

    model = ByteCNN().cpu()
    model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=False))
    model.eval()
    dummy = torch.zeros((1, BYTE_LENGTH), dtype=torch.long)
    torch.onnx.export(
        model,
        dummy,
        str(out_path),
        export_params=True,
        opset_version=18,
        do_constant_folding=True,
        input_names=["input_bytes"],
        output_names=["output"],
        dynamic_axes={"input_bytes": {0: "batch_size"}, "output": {0: "batch_size"}},
    )
    embedded = onnx.load(str(out_path), load_external_data=True)
    onnx.save_model(embedded, str(out_path), save_as_external_data=False)
    sidecar = out_path.with_suffix(out_path.suffix + ".data")
    if sidecar.is_file():
        sidecar.unlink()
    print(f"ONNX saved: {out_path} ({out_path.stat().st_size/1024:.1f} KB)")

In [ ]:
# ── Main: sequential training 2020 → 2021 → 2022 → 2023 ─────────────────────

checkpoint = MODELS / CHECKPOINT_NAME
model = ByteCNN().to(device)
continued = False

if checkpoint.is_file():
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=False))
    continued = True
    print(f"Resumed weights from {checkpoint}")

year_stats: List[YearStats] = []

for year in YEAR_ORDER:
    stats = train_one_year(model, year, checkpoint, continued=continued)
    year_stats.append(stats)
    continued = True
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=False))

    row = asdict(stats)
    pd.DataFrame([row]).to_csv(STATS / f"year_{year}.csv", index=False)
    print(f"Stats written: {STATS / f'year_{year}.csv'}")

summary = pd.DataFrame([asdict(s) for s in year_stats])
summary_path = STATS / "yearwise_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print(f"\nSummary: {summary_path}")

In [ ]:
# Export final ONNX (VigiDroid-compatible filename)
onnx_path = MODELS / ONNX_NAME
export_onnx(checkpoint, onnx_path)

# Kaggle Output tab will expose /kaggle/working/
print("\nArtifacts:")
for f in sorted(MODELS.glob("*")):
    print(f"  {f}  ({f.stat().st_size/1024:.1f} KB)")
print("\nDownload from Kaggle Output or add as Dataset for local vigidroid deploy:")
print(f"  cp {onnx_path} vigidroid/app/src/main/assets/bytecnn_basemodel_2020.onnx")

## Year-wise stats columns

| Column | Meaning |
|--------|---------|
| `benign_total` / `malware_total` | APK counts listed on HF for that year |
| `total_samples` | benign + malware |
| `val_samples` | held-out 10% downloaded for validation |
| `train_chunks` | streaming download batches processed |
| `epochs` | training epochs per year |
| `best_val_accuracy` / `best_val_loss` | best checkpoint during that year |
| `final_train_loss` / `final_val_loss` | last epoch metrics |
| `continued_from_checkpoint` | True if weights carried from prior year |

## Full 108 GB training notes

- Set `MAX_PER_CLASS = None` when you have enough Kaggle disk **or** run on a cloud VM with large volume.
- Default `DOWNLOAD_CHUNK=400` deletes APKs after each chunk to stay under ~20 GB disk.
- For **full** continual training without subsampling, consider Kaggle **Persistence** disk upgrade or run the same notebook on GCP/AWS with 200+ GB disk.
- After training, copy `bytecnn_basemodel_2020.onnx` into `vigidroid/app/src/main/assets/`.